# NudiLLM — Kannada Mini Language Model (Colab)

Train a GPT-style Kannada language model on Google Colab.

**Before running:** Runtime -> Change runtime type -> GPU (T4 or A100)

---
### Checklist
- [ ] GPU runtime selected
- [ ] Google Drive will be mounted for persistent checkpoints
- [ ] Repo will be cloned to `/content/nudi-llm`

## Step 0 — Check GPU

In [ ]:
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    bf16 = torch.cuda.is_bf16_supported()
    print("GPU :", name, round(vram, 1), "GB VRAM")
    print("BF16:", bf16)
    print("PyTorch:", torch.__version__)
else:
    print("No GPU! Go to Runtime -> Change runtime type -> GPU")

## Step 1 — Mount Google Drive
Checkpoints are saved here so they survive session resets.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
DRIVE_ROOT = "/content/drive/MyDrive/NudiLLM"
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("Drive mounted:", DRIVE_ROOT)

## Step 2 — Clone Repo & Install Dependencies

In [ ]:
import os, subprocess
REPO_DIR = "/content/nudi-llm"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone",
        "https://github.com/muhammadnavas/nudi-llm.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Dependencies installed")

## Step 3 — Restore from Drive (Resume Only)
Skip on a fresh run.

In [ ]:
import os, shutil
DRIVE_ROOT = "/content/drive/MyDrive/NudiLLM"
REPO_DIR   = "/content/nudi-llm"

drive_tok = os.path.join(DRIVE_ROOT, "tokenizer")
local_tok = os.path.join(REPO_DIR, "tokenizer")
if os.path.exists(drive_tok):
    if os.path.exists(local_tok):
        shutil.rmtree(local_tok)
    shutil.copytree(drive_tok, local_tok)
    print("Tokenizer restored from Drive")
else:
    print("No tokenizer on Drive")

for fname in ["train.tokens.npy", "val.tokens.npy"]:
    src = os.path.join(DRIVE_ROOT, "data", fname)
    dst = os.path.join(REPO_DIR, "data", "processed", fname)
    if os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        print("Restored", fname)
    else:
        print("Not on Drive:", fname)

## Step 4 — Download & Prepare Kannada Data
Skip if train.txt exists.

In [ ]:
import os, subprocess
TRAIN_TXT = "/content/nudi-llm/data/processed/train.txt"
if os.path.exists(TRAIN_TXT):
    size_mb = os.path.getsize(TRAIN_TXT) / 1024**2
    print("train.txt exists (", round(size_mb, 1), "MB) -- skipping")
else:
    subprocess.run(["python", "data/prepare_data.py"], check=True)

## Step 5 — Train BPE Tokenizer
Skip if tokenizer exists.

In [ ]:
import os, shutil, subprocess
TOK_MODEL  = "/content/nudi-llm/tokenizer/kannada_bpe.model"
DRIVE_ROOT = "/content/drive/MyDrive/NudiLLM"
if os.path.exists(TOK_MODEL):
    print("Tokenizer exists -- skipping")
else:
    subprocess.run(["python", "tokenizer/train_tokenizer.py"], check=True)
    drive_tok = os.path.join(DRIVE_ROOT, "tokenizer")
    if os.path.exists(drive_tok):
        shutil.rmtree(drive_tok)
    shutil.copytree("/content/nudi-llm/tokenizer", drive_tok)
    print("Tokenizer backed up to Drive:", drive_tok)

## Step 6 — Pre-tokenize Corpus
Runs once. Skip if .npy files exist.

In [ ]:
import os, shutil, subprocess
TRAIN_NPY  = "/content/nudi-llm/data/processed/train.tokens.npy"
DRIVE_ROOT = "/content/drive/MyDrive/NudiLLM"
if os.path.exists(TRAIN_NPY):
    import numpy as np
    n = len(np.load(TRAIN_NPY))
    print("Token cache exists,", n, "train tokens -- skipping")
else:
    subprocess.run(["python", "pretokenize.py"], check=True)
    drive_data = os.path.join(DRIVE_ROOT, "data")
    os.makedirs(drive_data, exist_ok=True)
    for fname in ["train.tokens.npy", "val.tokens.npy"]:
        src = "/content/nudi-llm/data/processed/" + fname
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(drive_data, fname))
            print(fname, "backed up to Drive")

## Step 7 — Train!

Colab config (auto-selected by `--colab` flag):
- **Batch size:** 16, effective 32 with grad accum
- **Context length:** 1024 tokens
- **Model:** 8 layers x 8 heads x 512 dim
- **Precision:** BF16 on A100, FP16 on T4
- **Checkpoints:** Drive every 500 steps

Estimated: ~30-60 min (T4) | ~10-20 min (A100)

In [ ]:
import subprocess
VERSION = "colab_v1"
RESUME  = False
cmd = ["python", "train.py", "--colab", "--version", VERSION]
if RESUME:
    cmd.append("--resume")
subprocess.run(cmd, check=True)

## Step 8 — Plot Training Loss

In [ ]:
import json
import matplotlib.pyplot as plt

VERSION  = "colab_v1"
log_path = "/content/drive/MyDrive/NudiLLM/logs/" + VERSION + "/train_log.json"

with open(log_path) as f:
    logs = json.load(f)

train_steps  = [d["step"] for d in logs["train"]]
train_losses = [d["loss"] for d in logs["train"]]
val_steps    = [d["step"] for d in logs["val"]]
val_losses   = [d["loss"] for d in logs["val"]]

plt.figure(figsize=(12, 5))
plt.plot(train_steps, train_losses, alpha=0.5, label="Train Loss")
plt.plot(val_steps, val_losses, linewidth=2, color="red", label="Val Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("NudiLLM Training Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
out_img = "/content/drive/MyDrive/NudiLLM/training_curve_" + VERSION + ".png"
plt.savefig(out_img, dpi=150)
plt.show()
print("Final train loss:", round(train_losses[-1], 4))
print("Final val   loss:", round(val_losses[-1], 4))

## Step 9 — Run Inference

In [ ]:
import sys, torch
sys.path.insert(0, "/content/nudi-llm")
from model.nudi import NudiLLM, NudiConfig
import sentencepiece as spm

VERSION      = "colab_v1"
CKPT_PATH    = "/content/drive/MyDrive/NudiLLM/checkpoints/" + VERSION + "/best.pt"
TOK_PATH     = "/content/nudi-llm/tokenizer/kannada_bpe.model"
PROMPT       = "\u0c95\u0cb0\u0ccd\u0ca8\u0cbe\u0c9f\u0c95"
MAX_NEW_TOKS = 100
TEMPERATURE  = 0.8
TOP_K        = 50

device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt  = torch.load(CKPT_PATH, map_location=device)
cfg   = NudiConfig(**ckpt["config"])
model = NudiLLM(cfg).to(device)
model.load_state_dict(ckpt["model"])
model.eval()
print("Model loaded | Val loss:", ckpt.get("val_loss", "N/A"))

sp = spm.SentencePieceProcessor()
sp.Load(TOK_PATH)

input_ids = torch.tensor([sp.encode(PROMPT)], dtype=torch.long).to(device)
with torch.no_grad():
    for _ in range(MAX_NEW_TOKS):
        logits, _ = model(input_ids[:, -cfg.context_length:])
        logits = logits[:, -1, :] / TEMPERATURE
        if TOP_K > 0:
            v, _ = torch.topk(logits, TOP_K)
            logits[logits < v[:, [-1]]] = float("-inf")
        probs  = torch.softmax(logits, dim=-1)
        next_t = torch.multinomial(probs, 1)
        input_ids = torch.cat([input_ids, next_t], dim=1)

generated = sp.decode(input_ids[0].tolist())
print("\n" + "="*60)
print(generated)
print("="*60)

## Step 10 — Download Best Checkpoint

In [ ]:
from google.colab import files
VERSION   = "colab_v1"
best_ckpt = "/content/drive/MyDrive/NudiLLM/checkpoints/" + VERSION + "/best.pt"
files.download(best_ckpt)